# 🌳 Lab W5-3 — Entropy, Information Gain และต้นไม้ที่อธิบายได้

**รายวิชาระบบสนับสนุนการตัดสินใจ · สัปดาห์ที่ 5 — Data Mining I**

Lab นี้ใช้คู่กับสื่อจำลอง **Decision Tree Grower** (`/sims/tree-grower`)
ตัวเลขที่คุณคำนวณได้ในสมุดเล่มนี้ต้องตรงกับตัวเลขบนหน้าจอสื่อจำลองทุกหลัก

## สิ่งที่จะได้เรียนรู้
1. คำนวณ **entropy** และ **information gain** ด้วยมือ ไม่ใช่เรียกไลบรารี
2. อธิบายว่าเหตุใดต้นไม้จึงเลือกตัวแปรที่มันเลือก
3. แสดงให้เห็นว่า **ความแม่นสูงไม่ได้แปลว่าโมเดลมีประโยชน์**
4. หา **จุดที่ต้นไม้เริ่มจดจำเสียงรบกวน** (overfitting) ด้วยตัวเลข

## ข้อมูล
`churn.csv` — ลูกค้าโทรคมนาคม 3,000 ราย พร้อมผลว่าเลิกใช้บริการหรือไม่

In [ ]:
import math

import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

URL = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
       "master/datasets/week05/churn.csv")
df = pd.read_csv(URL)

print(f"จำนวนลูกค้า     : {len(df):,}")
print(f"เลิกใช้บริการ   : {int(df.churned.sum()):,} ({df.churned.mean()*100:.2f}%)")
print(f"\nประเภทสัญญา:\n{df.contract_type.value_counts().to_string()}")
df.head(5)

## ส่วนที่ 1 — Entropy คืออะไรกันแน่

entropy วัด **ความไม่แน่นอน** ของกลุ่ม — สูงสุดเมื่อผสมกันครึ่งต่อครึ่ง
และเป็นศูนย์เมื่อทุกคนในกลุ่มเหมือนกันหมด

### 🧑‍💻 งานที่ 1
เขียนฟังก์ชัน `entropy(pos, n)` ที่รับจำนวนบวกและจำนวนทั้งหมด
แล้วคืนค่า entropy แบบฐานสอง (ต้องจัดการกรณี p = 0 และ p = 1 ให้คืน 0)

ทดสอบว่า `entropy(50, 100) == 1.0` และ `entropy(0, 100) == 0.0`
แล้วคำนวณ entropy ของข้อมูลทั้งชุด

*เฉลยที่ถูกต้อง: entropy ที่ราก = 0.9083*

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 2 — Information Gain

### 🧑‍💻 งานที่ 2
เขียนฟังก์ชัน `info_gain(data, mask)` ที่คืนค่า
information gain ของการแบ่งสองทางตามเงื่อนไข `mask`

สูตร: `IG = entropy(พ่อ) − [ (n_ซ้าย/n) × entropy(ซ้าย) + (n_ขวา/n) × entropy(ขวา) ]`

แล้วคำนวณ IG ของเงื่อนไข **ประเภทสัญญา = รายเดือน**

*เฉลยที่ถูกต้อง: IG = 0.1763 · ซ้าย 1,577 ราย (53.39%) · ขวา 1,423 ราย (9.07%)*

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 3 — จัดอันดับผู้สมัครทุกตัว

### 🧑‍💻 งานที่ 3
ประเมิน information gain ของเงื่อนไขทั้ง 11 ข้อด้านล่าง แล้วเรียงจากมากไปน้อย

ก่อนรันโค้ด **ให้เดาก่อน** ว่าเงื่อนไขใดจะมาเป็นอันดับหนึ่ง แล้วจดคำตอบไว้

*เฉลยที่ถูกต้อง: อันดับ 1 คือ ประเภทสัญญา = รายเดือน (0.1763)
อันดับสุดท้ายคือ ใบแจ้งหนี้อิเล็กทรอนิกส์ (0.0003)*

In [ ]:
# เขียนโค้ดของคุณที่นี่


In [ ]:
CANDIDATES = {
    "ประเภทสัญญา = รายเดือน": lambda d: d.contract_type == "รายเดือน",
    "ประเภทสัญญา = 2 ปี": lambda d: d.contract_type == "2 ปี",
    "อายุการใช้งาน ≤ 12 เดือน": lambda d: d.tenure_months <= 12,
    "อายุการใช้งาน ≤ 24 เดือน": lambda d: d.tenure_months <= 24,
    "อายุการใช้งาน ≤ 36 เดือน": lambda d: d.tenure_months <= 36,
    "ค่าบริการ > 1,000 บาท": lambda d: d.monthly_charge > 1000,
    "แจ้งปัญหา ≥ 2 ครั้ง": lambda d: d.support_tickets >= 2,
    "แจ้งปัญหา ≥ 3 ครั้ง": lambda d: d.support_tickets >= 3,
    "อินเทอร์เน็ต = ไฟเบอร์": lambda d: d.internet_type == "ไฟเบอร์",
    "ไม่ตัดบัญชีอัตโนมัติ": lambda d: d.auto_payment == "ไม่",
    "ใบแจ้งหนี้อิเล็กทรอนิกส์": lambda d: d.paperless_billing == "ใช่",
}

## ส่วนที่ 4 — ปลูกต้นไม้สองชั้น

### 🧑‍💻 งานที่ 4
แบ่งด้วยเงื่อนไขที่ดีที่สุดที่ราก แล้วหาเงื่อนไขที่ดีที่สุดของแต่ละกิ่ง
จากนั้นสร้างตารางแสดงใบทั้ง 4 ใบ พร้อมอัตราเลิกใช้บริการของแต่ละใบ

*เฉลยที่ถูกต้อง: กิ่ง "รายเดือน" แบ่งต่อด้วยอายุการใช้งาน ≤ 24 เดือน (gain 0.0549)
ได้ใบที่มีอัตราเลิกใช้ 72.78%*

In [ ]:
# เขียนโค้ดของคุณที่นี่


> **อย่าเพิ่งดีใจกับความแม่น** การทายว่า "อยู่ต่อ" ทุกรายได้ 67.63% ทันทีโดยไม่ต้องมีโมเดล
> ต้นไม้ที่เพิ่งปลูกดีขึ้นเพียงไม่กี่จุด
>
> **คุณค่าจริงของต้นไม้ต้นนี้ไม่ได้อยู่ที่ความแม่น** แต่อยู่ที่มันชี้กลุ่มลูกค้า
> ที่มีอัตราเลิกใช้ 72.78% ให้ทีมการตลาดไปทำงานต่อได้ —
> กลุ่มที่มีขนาดพอจะทำแคมเปญ และมีเหตุผลที่อธิบายให้ผู้บริหารฟังได้ในประโยคเดียว

## ส่วนที่ 5 — ความแม่นที่ไม่มีประโยชน์

### 🧑‍💻 งานที่ 5
คำนวณสำหรับใบที่มีอัตราเลิกใช้สูงที่สุด

1. ถ้าทำแคมเปญรักษาลูกค้าเฉพาะใบนี้ จะครอบคลุมลูกค้าที่จะเลิกใช้ทั้งหมดกี่เปอร์เซ็นต์ (recall)
2. ถ้าแคมเปญมีต้นทุน 200 บาท/ราย และรักษาลูกค้าไว้ได้ 1 ราย มีมูลค่า 3,000 บาท
   การทำแคมเปญกับใบนี้คุ้มหรือไม่ (สมมติแคมเปญได้ผล 30% ของผู้ที่จะเลิกใช้)
3. เทียบกับการทำแคมเปญกับลูกค้าทุกราย — แบบใดคุ้มกว่า
   **คำเตือน: คำตอบอาจไม่ใช่อย่างที่คุณคิด** ให้ตอบตามตัวเลขที่ได้จริง
4. ถ้างบประมาณถูกจำกัดให้ติดต่อลูกค้าได้เท่ากับขนาดของใบที่เสี่ยงที่สุดเท่านั้น
   คำตอบข้อ 3 เปลี่ยนไปหรือไม่ และส่วนต่างเป็นเท่าไร

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 6 — ต้นไม้ลึกแค่ไหนถึงจะพอ

### 🧑‍💻 งานที่ 6
ใช้ `DecisionTreeClassifier` ของ scikit-learn ปลูกต้นไม้ที่ความลึก 1 ถึง 20
แบ่งข้อมูล 70/30 แล้ววาดกราฟความแม่นของชุดฝึกกับชุดทดสอบเทียบกัน

หาจุดที่ทั้งสองเส้นเริ่มแยกจากกัน แล้วอธิบายว่าเกิดอะไรขึ้นที่จุดนั้น

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 7 — แปลต้นไม้เป็นภาษาคน

### 🧑‍💻 งานที่ 7 (เขียนเป็นข้อความ)

1. เขียนกฎที่ได้จากต้นไม้ความลึก 2 เป็น **ภาษาไทยธรรมดา** ไม่เกิน 4 บรรทัด
   ที่ทีมการตลาดอ่านแล้วลงมือทำได้ทันที
2. ต้นไม้บอกว่าลูกค้าสัญญารายเดือนที่ใช้บริการไม่ถึง 2 ปีเสี่ยงสูงมาก
   นี่เป็น **ความสัมพันธ์** หรือ **เหตุและผล** — และการแยกสองอย่างนี้สำคัญอย่างไร
   ต่อการออกแบบแคมเปญ
3. ถ้าบริษัทยกเลิกสัญญารายเดือนทั้งหมดตามคำแนะนำของต้นไม้ จะเกิดอะไรขึ้น
   และเหตุใดต้นไม้จึงไม่สามารถเตือนเรื่องนี้ได้

In [ ]:
# เขียนโค้ดของคุณที่นี่


---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — ฟังก์ชัน entropy ที่ผ่านการทดสอบ | 2 |
| งานที่ 2 — ฟังก์ชัน information gain ถูกต้อง | 3 |
| งานที่ 3 — จัดอันดับครบ 11 เงื่อนไขและเดาก่อนรัน | 3 |
| งานที่ 4 — ปลูกต้นไม้สองชั้นและสรุปใบทั้ง 4 | 3 |
| งานที่ 5 — วิเคราะห์ความคุ้มค่าของแคมเปญ | 3 |
| งานที่ 6 — กราฟความลึกกับ overfitting และคำอธิบาย | 3 |
| งานที่ 7 — แปลเป็นภาษาคน และแยกความสัมพันธ์ออกจากเหตุผล | 3 |
| **รวม** | **20** |

> 💡 ตัวเลขทุกตัวในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง `/sims/tree-grower`
> ถ้าไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด — ให้ย้อนกลับไปตรวจก่อนส่ง